# Direct Preference Alignment (DPO) of LLM with PyTorch FSDP and QLoRA on Amazon SageMaker HyperPod

## Lab 1 - Data preparation

In this notebook, we prepare the [nvidia/When2Call](https://huggingface.co/datasets/nvidia/When2Call) preference dataset (function-calling) to later align Qwen/Qwen3-4B-Instruct-2507 with DPO on SageMaker HyperPod (EKS).

## Prerequisites

### Install requirements

In [ ]:
%pip install -r ./requirements.txt --upgrade

***

## Visualize and upload the dataset

In this example, we are going to load [nvidia/When2Call](https://huggingface.co/datasets/nvidia/When2Call) dataset, an open-source dataset and model suite focused on enabling and improving function calling capabilities for large language models (LLMs).

In [ ]:
import datasets
from datasets import load_dataset

dataset = (
    load_dataset(
        "nvidia/When2Call", 
        "train_pref",
        split="train",
        streaming=True,
    )
    .take(1000)
    .shuffle(buffer_size=500)
)

dataset = datasets.Dataset.from_generator(lambda: dataset, features=dataset.features)

dataset

In [ ]:
import pandas as pd

df = pd.DataFrame(dataset)

df.head()

Split the dataset into train, validation, and test

In [ ]:
from sklearn.model_selection import train_test_split

_, val = train_test_split(df, test_size=0.1, random_state=42)
train, test = train_test_split(_, test_size=0.01, random_state=42)

print("Number of train elements: ", len(train))
print("Number of validation elements: ", len(val))
print("Number of test elements: ", len(test))

Utility functions to format messages

In [ ]:
import json                                                                                                                                         
import re                                                                                                                                                                                                                                                                     

# --- Type mapping for tool definition conversion ---

TYPE_MAP = {
    "str": "string",
    "int": "integer",
    "float": "number",
    "bool": "boolean",
    "list": "array",
    "dict": "object",
}


def normalize_type(raw_type: str):
    """Convert 'str, optional' / 'dict' to JSON Schema type string."""
    base = raw_type.split(",")[0].strip()
    return TYPE_MAP.get(base, base)


def convert_tool_definition(tool_str):
    """Convert a When2Call tool JSON string to OpenAI-style tool schema."""
    tool = json.loads(tool_str) if isinstance(tool_str, str) else tool_str

    properties = {}
    required = []

    for prop_name, prop_def in tool.get("parameters", {}).get("properties", {}).items():
        schema_type = normalize_type(prop_def.get("type", "string"))
        prop_schema = {"type": schema_type}

        if "description" in prop_def:
            prop_schema["description"] = prop_def["description"]

        properties[prop_name] = prop_schema

        if "optional" not in prop_def.get("type", ""):
            required.append(prop_name)

    return {
        "type": "function",
        "function": {
            "name": tool["name"],
            "description": tool.get("description", ""),
            "parameters": {
                "type": "object",
                "properties": properties,
                **({"required": required} if required else {}),
            },
        },
    }


def parse_toolcall_response(content):
    """Parse <TOOLCALL>[...]</TOOLCALL> from a response string.

    Returns (is_tool_call, parsed):
    - (True,  list of {"type":"function","function":{"name":...,"arguments":...}})
    - (False, content_string)
    """
    match = re.search(r"<TOOLCALL>\s*(\[.*?\])\s*</TOOLCALL>", content, re.DOTALL)
    if not match:
        return False, content

    calls = json.loads(match.group(1))
    tool_calls = []
    for call in calls:
        tool_calls.append(
            {
                "type": "function",
                "function": {
                    "name": call["name"],
                    "arguments": json.dumps(call["arguments"]),
                },
            }
        )
    return True, tool_calls

Format the dataset into the messages format and then apply the model chat template

In [ ]:
import json
from datasets import Dataset
from tqdm import tqdm
from transformers import AutoTokenizer

model_id = "Qwen/Qwen3-4B-Instruct-2507"
tokenizer = AutoTokenizer.from_pretrained(model_id)

def prepare_dataset(sample):
    messages = []

    tools = [convert_tool_definition(t) for t in sample["tools"]]

    system_text = f"""
    These are the available tools:
    <tools>
    {json.dumps(tools, indent=2)}
    </tools>"""

    messages.append({
        "role": "system", "content": system_text
    })

    for message in sample["messages"]:
        if isinstance(message, str):
            message = json.loads(message)

        messages.append({"role": message["role"], "content": message["content"]})

    is_tool_call, parsed = parse_toolcall_response(sample["chosen_response"]["content"])
    if is_tool_call:
        chosen = {"role": "assistant", "content": "", "tool_calls": parsed}
    else:
        chosen = {"role": "assistant", "content": parsed, "tool_calls": []}

    is_tool_call, parsed = parse_toolcall_response(sample["rejected_response"]["content"])
    if is_tool_call:
        rejected = {"role": "assistant", "content": "", "tool_calls": parsed}
    else:
        rejected = {"role": "assistant", "content": parsed, "tool_calls": []}

    yield {
        "prompt": messages,
        "chosen": [chosen],
        "rejected": [rejected],
    }


def convert_to_messages(dataset):
    """Iteratively run conversion on multi-turn conversation and flatten to messages."""
    records = []

    for row in tqdm(dataset, total=len(dataset), desc="Converting to messages"):
        for example in prepare_dataset(row):
            records.append(example)
    # Convert list of dicts → Hugging Face Dataset and return
    return Dataset.from_list(records)

In [ ]:
from datasets import Dataset, DatasetDict
from random import randint

train_dataset = Dataset.from_pandas(train)
val_dataset = Dataset.from_pandas(val)
test_dataset = Dataset.from_pandas(test)

dataset_dpo = DatasetDict({"train": train_dataset, "val": val_dataset, "test": test_dataset})

train_dataset = convert_to_messages(dataset_dpo["train"])

print(
    json.dumps(
        train_dataset[randint(0, len(train_dataset) - 1)],
        indent=2,
    )
)

val_dataset = convert_to_messages(dataset_dpo["val"])
test_dataset = convert_to_messages(dataset_dpo["test"])

### Save the dataset

Save the train/validation splits under `./data` and the held-out test split as `./tmp/gen_qa.jsonl` (used by the evaluation notebook). Then copy `./data` to the shared FSx volume at `/data/shared/option-3-dpo/data` as described in the lab README.

In [ ]:
import os

os.makedirs("./data/train", exist_ok=True)
os.makedirs("./data/val", exist_ok=True)
os.makedirs("./tmp", exist_ok=True)

train_dataset.to_json("./data/train/dataset.jsonl", orient="records")
val_dataset.to_json("./data/val/dataset.jsonl", orient="records")
test_dataset.to_json("./tmp/gen_qa.jsonl", orient="records")

print("Saved:")
print("  ./data/train/dataset.jsonl")
print("  ./data/val/dataset.jsonl")
print("  ./tmp/gen_qa.jsonl (held-out test set for evaluation)")
print("\nNext: copy ./data to FSx at /data/shared/option-3-dpo/data (see README).")